### **Installations, Imports and Paths**

In [1]:
# ============================================================
# Cell 1 — Installations
# Baselines notebook: no LoRA, no training
# ============================================================

import importlib.util
import subprocess
import sys

def is_installed(import_name):
    return importlib.util.find_spec(import_name) is not None

required = {
    "unsloth": "unsloth",
    "unsloth_zoo": "unsloth_zoo",
    "bitsandbytes": "bitsandbytes",
    "sacrebleu": "sacrebleu",
    "evaluate": "evaluate",
    "datasets": "datasets",
    "transformers": "transformers",
    "accelerate": "accelerate",
}

missing = [
    pip_name
    for pip_name, import_name in required.items()
    if not is_installed(import_name)
]

print("Missing packages:", missing)

if missing:
    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        *missing,
    ]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)
else:
    print("All required packages are already installed.")

print("Installation finished.")

Missing packages: ['unsloth', 'unsloth_zoo', 'bitsandbytes', 'sacrebleu', 'evaluate']
Running: /usr/bin/python3 -m pip install -q --no-cache-dir unsloth unsloth_zoo bitsandbytes sacrebleu evaluate
Installation finished.


In [ ]:
# ============================================================
# Cell 2 — Imports and environment check
# ============================================================

import unsloth

import torch
import random
import numpy as np
import pandas as pd
import json
import re
import time
import gc

from pathlib import Path
from tqdm.auto import tqdm
from datasets import load_dataset, get_dataset_config_names, Dataset
import transformers
import accelerate
import sacrebleu

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("WARNING: CUDA is not available.")
    print("Baseline generation on CPU will be very slow.")

print("datasets:", Dataset)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("sacrebleu:", sacrebleu.__version__)
print("Environment check finished.")

In [ ]:
# ============================================================
# Cell 3 — Mount Drive and define paths
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/alexandria_qwen35_sft")

# Reuse the same prepared_data folder as the finetuning notebooks.
DATA_DIR = PROJECT_DIR / "prepared_data"

# Keep baseline predictions separate.
BASELINE_DIR = PROJECT_DIR / "baselines"
BASELINE_PRED_DIR = BASELINE_DIR / "predictions"

# Finetuned experiment reports are still here.
FINETUNED_PRED_DIR = PROJECT_DIR / "predictions"

for p in [DATA_DIR, BASELINE_DIR, BASELINE_PRED_DIR, FINETUNED_PRED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR:", DATA_DIR)
print("BASELINE_DIR:", BASELINE_DIR)
print("BASELINE_PRED_DIR:", BASELINE_PRED_DIR)
print("FINETUNED_PRED_DIR:", FINETUNED_PRED_DIR)

### **Baseline configuration**

In [ ]:
# ============================================================
# Cell 4 — Baseline configuration
# Same dataset/prompt config as finetuning notebooks
# ============================================================

SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATASET_NAME = "UBC-NLP/alexandria"

SELECTED_CONFIGS_MODE = "EG_ONLY"
MANUAL_CONFIGS = ["EG"]

MAX_CONTEXT_TURNS = 3
USE_PREVIOUS_ENGLISH_CONTEXT = True
USE_METADATA = True

MAX_SEQ_LENGTH = 768

# Evaluate one baseline at a time to avoid Colab memory issues.
# Run once with qwen35_2b_base, then rerun model/eval cells with gemma4_e2b_it_base.
# ACTIVE_BASELINE_LABEL = "qwen35_2b_base"
ACTIVE_BASELINE_LABEL = "gemma4_e2b_it_base"

BASELINE_MODELS = {
    "qwen35_2b_base": {
        "label": "qwen35_2b_base",
        "model_name": "unsloth/Qwen3.5-2B-Base",
        "template_mode": "qwen_manual",
        "load_in_4bit": False,
        "load_in_16bit": False,
        "max_new_tokens": 120,
        "repetition_penalty": 1.05,
        "no_repeat_ngram_size": None,
    },
    "gemma4_e2b_it_base": {
        "label": "gemma4_e2b_it_base",
        "model_name": "unsloth/gemma-4-E2B-it",
        "template_mode": "gemma_native",
        "load_in_4bit": True,
        "load_in_16bit": False,
        "max_new_tokens": 96,
        "repetition_penalty": 1.10,
        "no_repeat_ngram_size": 4,
    },
}

BASELINE_CFG = BASELINE_MODELS[ACTIVE_BASELINE_LABEL]

MODEL_NAME = BASELINE_CFG["model_name"]
EXPERIMENT_NAME = (
    f"baseline_no_finetune_{ACTIVE_BASELINE_LABEL}_"
    f"alexandria_{SELECTED_CONFIGS_MODE.lower()}_context{MAX_CONTEXT_TURNS}"
)

print("Active baseline:", ACTIVE_BASELINE_LABEL)
print("Model:", MODEL_NAME)
print("Experiment:", EXPERIMENT_NAME)
print("Template mode:", BASELINE_CFG["template_mode"])

### **Data Download and preparation**

In [ ]:
# ============================================================
# Cell 5 — List Alexandria configs and load selected configs
# Exact same config logic as finetuning notebooks
# ============================================================

available_configs = get_dataset_config_names(DATASET_NAME)

print("Available Alexandria configs:")
print(available_configs)

if SELECTED_CONFIGS_MODE == "EG_ONLY":
    selected_configs = ["EG"] if "EG" in available_configs else [available_configs[0]]

elif SELECTED_CONFIGS_MODE == "ALL":
    selected_configs = available_configs

elif SELECTED_CONFIGS_MODE == "MANUAL":
    selected_configs = MANUAL_CONFIGS
    missing = [c for c in selected_configs if c not in available_configs]
    if missing:
        raise ValueError(f"These configs are not available: {missing}")

else:
    raise ValueError("SELECTED_CONFIGS_MODE must be EG_ONLY, ALL, or MANUAL.")

print("\nSelected configs:")
print(selected_configs)

loaded = {}

for cfg in selected_configs:
    print(f"\nLoading config: {cfg}")
    ds_train = load_dataset(DATASET_NAME, name=cfg, split="train")
    ds_test  = load_dataset(DATASET_NAME, name=cfg, split="test")

    loaded[cfg] = {
        "train": ds_train,
        "test": ds_test,
    }

    print("Train:", ds_train)
    print("Test:", ds_test)
    print("Example keys:", ds_train[0].keys())

In [ ]:
# ============================================================
# Cell 6 — Helper functions for robust extraction
# Exact same as finetuning notebooks
# ============================================================

def safe_get(row, keys, default=""):
    for k in keys:
        if isinstance(row, dict) and k in row and row[k] is not None:
            return row[k]
    return default

def turn_text(turn):
    if isinstance(turn, dict):
        for k in ["text", "sentence", "utterance", "content", "value"]:
            if k in turn and turn[k] is not None:
                return str(turn[k]).strip()
        return str(turn).strip()
    return str(turn).strip()

def turn_field(turn, keys, default=""):
    if isinstance(turn, dict):
        for k in keys:
            if k in turn and turn[k] is not None:
                return str(turn[k]).strip()
    return default

def normalize_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return list(x)

def truncate_text(text, max_chars=1200):
    text = str(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + " ..."

In [ ]:
# ============================================================
# Cell 7 — Flatten Alexandria conversations
# Exact same logic as finetuning notebooks
# ============================================================

def flatten_alexandria_split(ds, cfg_name, split_name, max_context_turns=3):
    records = []

    for conv_idx, row in enumerate(ds):
        english_conv = normalize_list(row["english_conversation"])
        dialect_conv = normalize_list(row["dialectal_conversation"])

        n = min(len(english_conv), len(dialect_conv))

        country = safe_get(row, ["country", "country_code"], cfg_name)
        dialect = safe_get(row, ["dialect", "dialect_label", "subdialect", "city", "variety"], "")
        domain = safe_get(row, ["domain", "topic"], "")
        persona = safe_get(row, ["persona", "roles", "speaker_roles"], "")
        conv_id = safe_get(row, ["conversation_id", "id", "dialogue_id"], f"{cfg_name}_{split_name}_{conv_idx}")

        for i in range(n):
            en_turn = english_conv[i]
            ar_turn = dialect_conv[i]

            source_text = turn_text(en_turn)
            target_text = turn_text(ar_turn)

            if not source_text or not target_text:
                continue

            prev_start = max(0, i - max_context_turns)
            prev_en_turns = english_conv[prev_start:i]

            previous_context = []
            for j, t in enumerate(prev_en_turns, start=prev_start):
                previous_context.append({
                    "turn_id": j,
                    "speaker": turn_field(t, ["speaker", "role", "speaker_role"], ""),
                    "direction": turn_field(t, ["direction", "gender_direction", "speaker_addressee_gender"], ""),
                    "text": turn_text(t),
                })

            records.append({
                "source_id": f"{cfg_name}_{split_name}_{conv_id}_{i}",
                "config": cfg_name,
                "split": split_name,
                "conversation_id": conv_id,
                "turn_id": i,

                "country": country,
                "dialect": dialect,
                "domain": domain,
                "persona": persona,

                "speaker": turn_field(en_turn, ["speaker", "role", "speaker_role"], ""),
                "gender_direction": turn_field(en_turn, ["direction", "gender_direction", "speaker_addressee_gender"], ""),

                "previous_english_turns": previous_context,
                "source_text": source_text,
                "target_arabic": target_text,
            })

    return records

train_records = []
eval_records = []

for cfg in selected_configs:
    train_records.extend(
        flatten_alexandria_split(
            loaded[cfg]["train"],
            cfg_name=cfg,
            split_name="train",
            max_context_turns=MAX_CONTEXT_TURNS,
        )
    )

    eval_records.extend(
        flatten_alexandria_split(
            loaded[cfg]["test"],
            cfg_name=cfg,
            split_name="test",
            max_context_turns=MAX_CONTEXT_TURNS,
        )
    )

train_df = pd.DataFrame(train_records)
eval_df = pd.DataFrame(eval_records)

print("Train shape:", train_df.shape)
print("Eval shape:", eval_df.shape)

print("\nTrain configs:")
print(train_df["config"].value_counts())

print("\nEval configs:")
print(eval_df["config"].value_counts())

display(train_df.head())

In [ ]:
# ============================================================
# Cell 8 — Save prepared flattened data
# Same filenames as finetuning notebooks
# ============================================================

train_jsonl = DATA_DIR / f"alexandria_train_{SELECTED_CONFIGS_MODE.lower()}_context{MAX_CONTEXT_TURNS}.jsonl"
eval_jsonl  = DATA_DIR / f"alexandria_eval_{SELECTED_CONFIGS_MODE.lower()}_context{MAX_CONTEXT_TURNS}.jsonl"

train_df.to_json(train_jsonl, orient="records", lines=True, force_ascii=False)
eval_df.to_json(eval_jsonl, orient="records", lines=True, force_ascii=False)

print("Saved train:", train_jsonl)
print("Saved eval:", eval_jsonl)

### **Building prompts**

In [ ]:
# ============================================================
# Cell 9 — Build prompt and chat messages
# Exact same as finetuning notebooks
# ============================================================

SYSTEM_PROMPT = (
    "You are a professional machine translation system. "
    "Translate the current English dialogue turn into natural dialectal Arabic. "
    "Return only the translation, without explanation."
)

def build_context(previous_turns):
    if not USE_PREVIOUS_ENGLISH_CONTEXT or not previous_turns:
        return "No previous context."

    lines = []
    for i, t in enumerate(previous_turns, start=1):
        speaker = t.get("speaker", "")
        text = t.get("text", "")
        if speaker:
            lines.append(f"{i}. {speaker}: {text}")
        else:
            lines.append(f"{i}. {text}")

    return "\n".join(lines)

def build_metadata_block(row):
    if not USE_METADATA:
        return "No metadata."

    fields = [
        ("Country/config", row.get("config", "")),
        ("Target dialect", row.get("dialect", "")),
        ("Domain", row.get("domain", "")),
        ("Persona/Roles", row.get("persona", "")),
        ("Current speaker", row.get("speaker", "")),
        ("Speaker-to-addressee gender direction", row.get("gender_direction", "")),
    ]

    lines = []
    for k, v in fields:
        v = str(v).strip()
        if v:
            lines.append(f"{k}: {v}")

    return "\n".join(lines) if lines else "No metadata."

def make_user_prompt(row):
    context = build_context(row["previous_english_turns"])
    metadata = build_metadata_block(row)

    return f"""Task:
Translate the current English dialogue turn into the target dialectal Arabic variety.

Metadata:
{metadata}

Previous English dialogue context:
{context}

Current English turn:
{row["source_text"]}

Rules:
- Preserve the meaning exactly.
- Use natural local dialectal Arabic, not Modern Standard Arabic unless it is natural in context.
- Preserve names, numbers, named entities, and technical terms when appropriate.
- Keep the tone appropriate for the speaker and domain.
- Return only the Arabic translation."""

def row_to_messages(row):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(row)},
        {"role": "assistant", "content": row["target_arabic"]},
    ]

train_df["messages"] = train_df.apply(row_to_messages, axis=1)
eval_df["messages"]  = eval_df.apply(row_to_messages, axis=1)

train_dataset = Dataset.from_pandas(
    train_df[
        [
            "messages",
            "source_id",
            "config",
            "domain",
            "dialect",
            "source_text",
            "target_arabic",
        ]
    ],
    preserve_index=False,
)

eval_dataset = Dataset.from_pandas(
    eval_df[
        [
            "messages",
            "source_id",
            "config",
            "domain",
            "dialect",
            "source_text",
            "target_arabic",
        ]
    ],
    preserve_index=False,
)

print(train_dataset)
print(eval_dataset)

print("\nExample messages:")
train_dataset[0]["messages"]

Sanity Checks

In [ ]:
# ============================================================
# Cell 10 — Optional faithfulness check
# Confirms current eval_df has the same source_ids as existing finetuned prediction files
# ============================================================

def find_latest_csv(pattern, folder):
    files = sorted(folder.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    return files[0] if files else None

expected_ids = set(eval_df["source_id"].astype(str).tolist())

REFERENCE_FINETUNED_EXPERIMENTS = [
    "qwen35_2b_alexandria_eg_only_context3_all_group_r16_v2_10epochs",
    "gemma4_e2b_it_alexandria_eg_only_context3_all_group_r16_native_template_eos_4bit_lr5e5_resume_safe_v2",
]

for exp_name in REFERENCE_FINETUNED_EXPERIMENTS:
    pred_file = find_latest_csv(f"full_eval_predictions_{exp_name}_best_step*.csv", FINETUNED_PRED_DIR)

    print("\nChecking:", exp_name)

    if pred_file is None:
        print("No finetuned prediction file found yet. Skipping.")
        continue

    ref_df = pd.read_csv(pred_file)

    if "source_id" not in ref_df.columns:
        print("Prediction file has no source_id column:", pred_file)
        continue

    actual_ids = set(ref_df["source_id"].astype(str).tolist())

    print("Prediction file:", pred_file)
    print("Current eval examples:", len(expected_ids))
    print("Prediction source_ids:", len(actual_ids))
    print("Missing in prediction:", len(expected_ids - actual_ids))
    print("Extra in prediction:", len(actual_ids - expected_ids))

    if expected_ids == actual_ids:
        print("PASS: source_id set is exactly identical.")
    else:
        print("WARNING: source_id set differs.")

### **Preparing baseline models**

In [ ]:
# ============================================================
# Cell 11 — Load active baseline model
# No LoRA adapter, no finetuning
# ============================================================

try:
    from unsloth import FastLanguageModel
    UnslothModel = FastLanguageModel
    print("Using unsloth.FastLanguageModel")
except Exception as e:
    print("FastLanguageModel import failed:", repr(e))
    from unsloth import FastModel
    UnslothModel = FastModel
    print("Using unsloth.FastModel fallback")

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

if BASELINE_CFG["load_in_4bit"]:
    dtype = None
else:
    dtype = torch.bfloat16 if USE_BF16 else torch.float16

def load_baseline_model():
    kwargs = dict(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
    )

    try:
        return UnslothModel.from_pretrained(
            **kwargs,
            dtype=dtype,
            load_in_4bit=BASELINE_CFG["load_in_4bit"],
            load_in_16bit=BASELINE_CFG["load_in_16bit"],
        )
    except TypeError:
        return UnslothModel.from_pretrained(
            **kwargs,
            dtype=dtype,
            load_in_4bit=BASELINE_CFG["load_in_4bit"],
        )

model, tokenizer = load_baseline_model()

if hasattr(tokenizer, "tokenizer"):
    print("Tokenizer object has internal tokenizer. Using tokenizer.tokenizer.")
    tokenizer = tokenizer.tokenizer

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

STOP_TOKEN_IDS = []
if tokenizer.eos_token_id is not None:
    STOP_TOKEN_IDS.append(int(tokenizer.eos_token_id))

for tok in ["<end_of_turn>", "<|im_end|>", "<turn|>"]:
    try:
        tok_id = tokenizer.convert_tokens_to_ids(tok)
        if tok_id is not None and tok_id != tokenizer.unk_token_id and tok_id not in STOP_TOKEN_IDS:
            STOP_TOKEN_IDS.append(int(tok_id))
    except Exception:
        pass

if len(STOP_TOKEN_IDS) == 0:
    raise RuntimeError("Could not identify EOS/stop token id.")

try:
    UnslothModel.for_inference(model)
except Exception as e:
    print("for_inference not available or not needed:", repr(e))

print("Loaded baseline model:", MODEL_NAME)
print("dtype:", dtype)
print("load_in_4bit:", BASELINE_CFG["load_in_4bit"])
print("Tokenizer type:", type(tokenizer))
print("pad token:", tokenizer.pad_token, tokenizer.pad_token_id)
print("eos token:", tokenizer.eos_token, tokenizer.eos_token_id)
print("stop token ids:", STOP_TOKEN_IDS)
print("chat template exists:", getattr(tokenizer, "chat_template", None) is not None)

In [ ]:
# ============================================================
# Cell 12 — Model-specific inference formatting
# Qwen uses exact manual SFT template from Qwen notebook
# Gemma uses exact native template style from Gemma notebook
# ============================================================

SYSTEM_MARKER = "### System:"
INSTRUCTION_MARKER = "### Instruction:"
RESPONSE_MARKER = "### Arabic translation:"

def format_sft_text(system_text, user_text, assistant_text=None, add_eos=True):
    text = (
        f"{SYSTEM_MARKER}\n"
        f"{system_text.strip()}\n\n"
        f"{INSTRUCTION_MARKER}\n"
        f"{user_text.strip()}\n\n"
        f"{RESPONSE_MARKER}\n"
    )

    if assistant_text is not None:
        text += assistant_text.strip()

        if add_eos and tokenizer.eos_token is not None:
            text += tokenizer.eos_token

    return text

def extract_qwen_answer(decoded_text):
    if RESPONSE_MARKER in decoded_text:
        answer = decoded_text.split(RESPONSE_MARKER)[-1]
    else:
        answer = decoded_text

    special_tokens = [
        tokenizer.eos_token,
        tokenizer.pad_token,
        "<|endoftext|>",
        "<|im_end|>",
    ]

    for tok in special_tokens:
        if tok:
            answer = answer.replace(tok, "")

    return answer.strip()

def messages_to_gemma_chat(messages, include_assistant=False):
    system_text = ""
    user_text = ""
    assistant_text = ""

    for m in messages:
        role = m.get("role", "")
        content = str(m.get("content", ""))

        if role == "system":
            system_text = content.strip()
        elif role == "user":
            user_text = content.strip()
        elif role in ["assistant", "model"]:
            assistant_text = content.strip()

    folded_user = (
        "System instruction:\n"
        f"{system_text}\n\n"
        "User request:\n"
        f"{user_text}"
    ).strip()

    out = [{"role": "user", "content": folded_user}]

    if include_assistant:
        out.append({"role": "assistant", "content": assistant_text})

    return out

def apply_chat_template_robust(messages, tokenize=False, add_generation_prompt=False):
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=tokenize,
            add_generation_prompt=add_generation_prompt,
        )
    except Exception as e1:
        converted = []
        for m in messages:
            mm = dict(m)
            if mm.get("role") == "assistant":
                mm["role"] = "model"
            converted.append(mm)

        try:
            return tokenizer.apply_chat_template(
                converted,
                tokenize=tokenize,
                add_generation_prompt=add_generation_prompt,
            )
        except Exception as e2:
            print("apply_chat_template failed with assistant role:", repr(e1))
            print("apply_chat_template failed with model role:", repr(e2))
            raise

def clean_generation_text(text):
    text = str(text)

    for tok in [
        tokenizer.eos_token,
        tokenizer.pad_token,
        "<end_of_turn>",
        "<start_of_turn>",
        "<|endoftext|>",
        "<|im_end|>",
        "<turn|>",
    ]:
        if tok:
            text = text.replace(tok, "")

    for marker in ["assistant", "model"]:
        if text.strip().startswith(marker):
            text = text.strip()[len(marker):].strip()

    return text.strip()

def build_prompt_for_row(row):
    template_mode = BASELINE_CFG["template_mode"]

    if template_mode == "qwen_manual":
        user_text = make_user_prompt(row)

        prompt = format_sft_text(
            system_text=SYSTEM_PROMPT,
            user_text=user_text,
            assistant_text=None,
            add_eos=False,
        )

        return prompt

    if template_mode == "gemma_native":
        train_like_messages = row_to_messages({
            **row,
            "target_arabic": "",
        })

        gemma_messages = messages_to_gemma_chat(
            train_like_messages,
            include_assistant=False,
        )

        prompt = apply_chat_template_robust(
            gemma_messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        return prompt

    raise ValueError(f"Unknown template_mode: {template_mode}")

def generate_translation_from_row(row):
    prompt = build_prompt_for_row(row)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(model.device)

    input_len = inputs["input_ids"].shape[-1]

    eos_arg = STOP_TOKEN_IDS if len(STOP_TOKEN_IDS) > 1 else STOP_TOKEN_IDS[0]

    gen_kwargs = dict(
        **inputs,
        max_new_tokens=BASELINE_CFG["max_new_tokens"],
        do_sample=False,
        num_beams=1,
        repetition_penalty=BASELINE_CFG["repetition_penalty"],
        eos_token_id=eos_arg,
        pad_token_id=tokenizer.pad_token_id,
        use_cache=True,
    )

    if BASELINE_CFG["no_repeat_ngram_size"] is not None:
        gen_kwargs["no_repeat_ngram_size"] = BASELINE_CFG["no_repeat_ngram_size"]

    with torch.no_grad():
        outputs = model.generate(**gen_kwargs)

    template_mode = BASELINE_CFG["template_mode"]

    if template_mode == "qwen_manual":
        decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)
        answer = extract_qwen_answer(decoded)
        return answer, decoded

    if template_mode == "gemma_native":
        generated_ids = outputs[0][input_len:]
        raw_answer = tokenizer.decode(generated_ids, skip_special_tokens=False)
        answer = clean_generation_text(raw_answer)
        full_decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)
        return answer, full_decoded

    raise ValueError(f"Unknown template_mode: {template_mode}")

sample = eval_df.sample(1, random_state=SEED).iloc[0].to_dict()
pred, raw = generate_translation_from_row(sample)

print("Baseline:", ACTIVE_BASELINE_LABEL)
print("Config:", sample["config"])
print("Dialect:", sample["dialect"])
print("Domain:", sample["domain"])
print("\nEnglish:")
print(sample["source_text"])
print("\nReference Arabic:")
print(sample["target_arabic"])
print("\nPrediction:")
print(pred)

### **Generate full eval predictions**

In [ ]:
# ============================================================
# Cell 13 — Generate predictions on FULL eval/test set
# Safe resume
# ============================================================

EVAL_LIMIT = None
SAVE_EVERY = 25
STORE_RAW_OUTPUT = False
FORCE_REGENERATE_PREDICTIONS = False

pred_path = BASELINE_PRED_DIR / f"full_eval_predictions_{EXPERIMENT_NAME}.csv"
tmp_pred_path = BASELINE_PRED_DIR / f"full_eval_predictions_{EXPERIMENT_NAME}.partial.csv"

print("Baseline:", ACTIVE_BASELINE_LABEL)
print("Model:", MODEL_NAME)
print("Experiment:", EXPERIMENT_NAME)
print("Saving predictions to:", pred_path)
print("Temporary partial file:", tmp_pred_path)

full_eval_df = eval_df.reset_index(drop=True).copy()

if EVAL_LIMIT is not None:
    full_eval_df = full_eval_df.iloc[:EVAL_LIMIT].copy()

expected_n = len(full_eval_df)
expected_ids = set(full_eval_df["source_id"].astype(str).tolist())

print("Total eval/test examples to evaluate:", expected_n)

resume_path = None

if FORCE_REGENERATE_PREDICTIONS:
    print("FORCE_REGENERATE_PREDICTIONS=True")
    pred_rows = []
    done_ids = set()

else:
    if tmp_pred_path.exists():
        resume_path = tmp_pred_path
    elif pred_path.exists():
        resume_path = pred_path

    if resume_path is not None:
        print("Found existing prediction file:")
        print(resume_path)

        existing_df = pd.read_csv(resume_path)

        required_cols = {
            "source_id",
            "prediction",
            "model_name",
            "experiment_name",
        }

        missing_cols = required_cols - set(existing_df.columns)

        if missing_cols:
            print("Existing prediction file is incompatible.")
            print("Missing columns:", missing_cols)
            print("Starting prediction from scratch.")
            pred_rows = []
            done_ids = set()
        else:
            existing_df["source_id"] = existing_df["source_id"].astype(str)
            existing_df = existing_df[existing_df["source_id"].isin(expected_ids)].copy()
            existing_df = existing_df.drop_duplicates(subset=["source_id"], keep="first").copy()

            pred_rows = existing_df.to_dict("records")
            done_ids = set(existing_df["source_id"].astype(str).tolist())

            print("Resuming prediction generation.")
            print("Already completed examples:", len(done_ids))
    else:
        print("No existing prediction file found. Starting from scratch.")
        pred_rows = []
        done_ids = set()

if len(done_ids) == expected_n:
    print("Prediction file already contains all expected eval examples.")
    pred_df = pd.DataFrame(pred_rows)
    pred_df.to_csv(pred_path, index=False, encoding="utf-8-sig")

    if tmp_pred_path.exists():
        try:
            tmp_pred_path.unlink()
            print("Removed partial file after confirming full completion.")
        except Exception as e:
            print("Could not remove partial file:", repr(e))

    display(pred_df.head())

else:
    print(f"Remaining examples to generate: {expected_n - len(done_ids)}")

    start_time = time.time()

    for _, row in tqdm(full_eval_df.iterrows(), total=len(full_eval_df)):
        row_dict = row.to_dict()
        source_id = str(row_dict["source_id"])

        if source_id in done_ids:
            continue

        try:
            pred, raw = generate_translation_from_row(row_dict)

            out_row = {
                "source_id": row_dict["source_id"],
                "config": row_dict.get("config", ""),
                "dialect": row_dict.get("dialect", ""),
                "domain": row_dict.get("domain", ""),
                "source_text": row_dict["source_text"],
                "reference_arabic": row_dict["target_arabic"],
                "prediction": pred,
                "model_name": MODEL_NAME,
                "experiment_name": EXPERIMENT_NAME,
                "baseline_label": ACTIVE_BASELINE_LABEL,
                "template_mode": BASELINE_CFG["template_mode"],
            }

            if STORE_RAW_OUTPUT:
                out_row["raw_output"] = raw

        except Exception as e:
            out_row = {
                "source_id": row_dict.get("source_id", ""),
                "config": row_dict.get("config", ""),
                "dialect": row_dict.get("dialect", ""),
                "domain": row_dict.get("domain", ""),
                "source_text": row_dict.get("source_text", ""),
                "reference_arabic": row_dict.get("target_arabic", ""),
                "prediction": "",
                "generation_error": repr(e),
                "model_name": MODEL_NAME,
                "experiment_name": EXPERIMENT_NAME,
                "baseline_label": ACTIVE_BASELINE_LABEL,
                "template_mode": BASELINE_CFG["template_mode"],
            }

        pred_rows.append(out_row)
        done_ids.add(source_id)

        if len(pred_rows) % SAVE_EVERY == 0:
            tmp_df = pd.DataFrame(pred_rows)
            tmp_df = tmp_df.drop_duplicates(subset=["source_id"], keep="first")
            tmp_df.to_csv(tmp_pred_path, index=False, encoding="utf-8-sig")
            print(f"Saved partial predictions: {len(tmp_df)} rows")

    pred_df = pd.DataFrame(pred_rows)
    pred_df = pred_df.drop_duplicates(subset=["source_id"], keep="first")
    pred_df.to_csv(pred_path, index=False, encoding="utf-8-sig")

    if tmp_pred_path.exists():
        try:
            tmp_pred_path.unlink()
            print("Removed partial prediction file after successful final save.")
        except Exception as e:
            print("Could not remove partial file:", repr(e))

    elapsed = time.time() - start_time

    actual_ids = set(pred_df["source_id"].astype(str).tolist())
    missing_ids = expected_ids - actual_ids
    extra_ids = actual_ids - expected_ids

    print("\nDone.")
    print("Saved predictions to:", pred_path)
    print("Total rows saved:", len(pred_df))
    print("Expected eval/test rows:", expected_n)
    print(f"Elapsed time: {elapsed / 60:.2f} minutes")

    if missing_ids:
        raise RuntimeError(f"Prediction file is incomplete. Missing {len(missing_ids)} eval examples.")

    if extra_ids:
        print(f"Warning: prediction file has {len(extra_ids)} extra source_ids not in current eval_df.")

    if len(pred_df) == expected_n:
        print("Full eval/test set was evaluated successfully.")

    display(pred_df.head())

### Free GPU before re-run

In [ ]:
# ============================================================
# Cell 15 — Free memory before switching ACTIVE_BASELINE_LABEL
# Run this, then go back to Cell 4, switch ACTIVE_BASELINE_LABEL, and rerun Cells 11–14
# ============================================================

try:
    del model
    del tokenizer
except Exception:
    pass

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("Memory cleaned. You can now switch ACTIVE_BASELINE_LABEL and run the next baseline.")

---

### **final Comparisons with Baselines**
##### **Measuring BLEU, ChrF, and similarity scores**

Run after compute all predictions

In [1]:
# ============================================================
# Cell 1 — Install required packages for semantic similarity
# ============================================================

import importlib.util
import subprocess
import sys

required = {
    "sentence_transformers": "sentence-transformers",
    "sklearn": "scikit-learn",
    "tqdm": "tqdm",
}

missing = []

for import_name, pip_name in required.items():
    if importlib.util.find_spec(import_name) is None:
        missing.append(pip_name)

print("Missing packages:", missing)

if missing:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        *missing
    ])
else:
    print("All required packages already installed.")

Missing packages: []
All required packages already installed.


In [2]:
! pip install -q --no-cache-dir sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 38.6 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# ------------------------------------------------------------
# Imports
# ------------------------------------------------------------

from pathlib import Path
import json
import gc
import pandas as pd
import numpy as np
import torch
import sacrebleu
from sentence_transformers import SentenceTransformer

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROJECT_DIR = Path("/content/drive/MyDrive/alexandria_qwen35_sft")

FINETUNED_PRED_DIR = PROJECT_DIR / "predictions"
BASELINE_PRED_DIR  = PROJECT_DIR / "baselines" / "predictions"

QWEN_METRICS_PATH = FINETUNED_PRED_DIR / (
    "full_eval_metrics_"
    "qwen35_2b_alexandria_eg_only_context3_all_group_r16_v2_10epochs"
    "_best_step600.json"
)

# Updated Gemma path
GEMMA_METRICS_PATH = Path(
    "/content/drive/MyDrive/alexandria_qwen35_sft/predictions/"
    "full_eval_metrics_gemma4_e2b_it_alexandria_eg_only_context3_"
    "fnn_group_r8_prompt_nat_eg_mlp_r8_lr1e5_10epochs_v1_best_step2500.json"
)

SEMANTIC_OUT_DIR = PROJECT_DIR / "semantic_similarity_e5_large"
SEMANTIC_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("QWEN_METRICS_PATH:", QWEN_METRICS_PATH)
print("GEMMA_METRICS_PATH:", GEMMA_METRICS_PATH)
print("BASELINE_PRED_DIR:", BASELINE_PRED_DIR)
print("SEMANTIC_OUT_DIR:", SEMANTIC_OUT_DIR)

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

E5_MODEL_NAME = "intfloat/multilingual-e5-large"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# E5-large is heavier, so keep this conservative.
BATCH_SIZE = 8

print("Using device:", DEVICE)
print("E5 model:", E5_MODEL_NAME)
print("Batch size:", BATCH_SIZE)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def read_json(path):
    path = Path(path)
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8"))


def derive_prediction_path_from_metrics_path(metrics_path):
    metrics_path = Path(metrics_path)
    pred_name = metrics_path.name.replace("full_eval_metrics_", "full_eval_predictions_")
    pred_name = pred_name.replace(".json", ".csv")
    return metrics_path.with_name(pred_name)


def get_prediction_file_from_metrics(metrics_path):
    metrics_path = Path(metrics_path)
    data = read_json(metrics_path)

    if data is None:
        derived = derive_prediction_path_from_metrics_path(metrics_path)
        if derived.exists():
            return derived, {}
        return None, None

    pred_file = data.get("prediction_file", None)

    if pred_file is not None and Path(pred_file).exists():
        return Path(pred_file), data

    derived = derive_prediction_path_from_metrics_path(metrics_path)

    if derived.exists():
        return derived, data

    return None, data


def safe_first(df, col, default=""):
    if col not in df.columns:
        return default

    vals = df[col].dropna().astype(str).unique()

    if len(vals) == 0:
        return default

    if len(vals) == 1:
        return vals[0]

    return list(vals)


def load_prediction_df(pred_path):
    pred_path = Path(pred_path)

    df = pd.read_csv(pred_path)

    required = {"source_id", "reference_arabic", "prediction"}
    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            f"Missing required columns in {pred_path}: {missing}\n"
            f"Available columns: {list(df.columns)}"
        )

    df["source_id"] = df["source_id"].astype(str)
    df["reference_arabic"] = df["reference_arabic"].fillna("").astype(str)
    df["prediction"] = df["prediction"].fillna("").astype(str)

    df = df.drop_duplicates(subset=["source_id"], keep="first").copy()
    df = df.reset_index(drop=True)

    return df


def compute_mt_metrics_from_df(df):
    preds = df["prediction"].fillna("").astype(str).tolist()
    refs  = df["reference_arabic"].fillna("").astype(str).tolist()

    bleu   = sacrebleu.corpus_bleu(preds, [refs])
    chrf   = sacrebleu.corpus_chrf(preds, [refs], word_order=0)
    chrfpp = sacrebleu.corpus_chrf(preds, [refs], word_order=2)

    return {
        "BLEU": float(bleu.score),
        "chrF": float(chrf.score),
        "chrF++": float(chrfpp.score),
    }


def e5_format(text):
    # Symmetric similarity: same prefix for reference and prediction.
    return "passage: " + str(text).strip()


def compute_e5_pairwise_similarity(df, model):
    refs = [e5_format(x) for x in df["reference_arabic"].tolist()]
    preds = [e5_format(x) for x in df["prediction"].tolist()]

    ref_emb = model.encode(
        refs,
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True,
    )

    pred_emb = model.encode(
        preds,
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True,
    )

    # Because embeddings are normalized, dot product = cosine similarity.
    sims = np.sum(ref_emb * pred_emb, axis=1)

    return sims


def summarize_similarity(scores):
    scores = np.asarray(scores, dtype=float)

    return {
        "E5_large_cosine_mean": float(np.mean(scores)),
        "E5_large_cosine_median": float(np.median(scores)),
        "E5_large_cosine_std": float(np.std(scores)),
        "E5_large_cosine_min": float(np.min(scores)),
        "E5_large_cosine_max": float(np.max(scores)),
        "E5_large_cosine_p05": float(np.percentile(scores, 5)),
        "E5_large_cosine_p25": float(np.percentile(scores, 25)),
        "E5_large_cosine_p75": float(np.percentile(scores, 75)),
        "E5_large_cosine_p95": float(np.percentile(scores, 95)),
    }


def compute_group_metrics(df, group_col):
    if group_col not in df.columns:
        return []

    rows = []

    for value in sorted(df[group_col].dropna().astype(str).unique()):
        tmp = df[df[group_col].astype(str) == value].copy()

        if len(tmp) == 0:
            continue

        mt_scores = compute_mt_metrics_from_df(tmp)
        semantic_scores = summarize_similarity(tmp["E5_large_cosine"].values)

        rows.append({
            group_col: value,
            "num_examples": int(len(tmp)),
            **mt_scores,
            **semantic_scores,
        })

    return rows


def make_exp_record_from_metrics(label, model_type, metrics_path):
    pred_file, metrics_data = get_prediction_file_from_metrics(metrics_path)

    if metrics_data is None:
        return {
            "label": label,
            "model_type": model_type,
            "metrics_path": str(metrics_path),
            "prediction_file": None,
            "metrics_data": {},
            "found": False,
            "error": "Metrics JSON not found and derived prediction CSV not found.",
        }

    if pred_file is None:
        return {
            "label": label,
            "model_type": model_type,
            "metrics_path": str(metrics_path),
            "prediction_file": None,
            "metrics_data": metrics_data,
            "found": False,
            "error": "Prediction file not found from JSON or derived path.",
        }

    return {
        "label": label,
        "model_type": model_type,
        "metrics_path": str(metrics_path),
        "prediction_file": str(pred_file),
        "metrics_data": metrics_data,
        "found": True,
        "error": "",
    }


# ------------------------------------------------------------
# Collect experiments
# ------------------------------------------------------------

experiments = []

experiments.append(
    make_exp_record_from_metrics(
        label="Qwen3.5-2B LoRA all-r16 best_step600",
        model_type="finetuned_lora",
        metrics_path=QWEN_METRICS_PATH,
    )
)

experiments.append(
    make_exp_record_from_metrics(
        label="Gemma-4-E2B-it FNN-r8 MLP best_step2500",
        model_type="finetuned_lora",
        metrics_path=GEMMA_METRICS_PATH,
    )
)

baseline_pred_files = sorted(
    p for p in BASELINE_PRED_DIR.glob("full_eval_predictions_baseline_no_finetune_*.csv")
    if ".partial" not in p.name
)

for p in baseline_pred_files:
    experiments.append({
        "label": p.stem.replace("full_eval_predictions_", ""),
        "model_type": "baseline_no_finetune",
        "metrics_path": None,
        "prediction_file": str(p),
        "metrics_data": {},
        "found": True,
        "error": "",
    })

print("\nExperiments to evaluate:")
for e in experiments:
    print("-", e["label"], "| found:", e["found"], "| pred:", e["prediction_file"])

# ------------------------------------------------------------
# Load E5-large
# ------------------------------------------------------------

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

gc.collect()

e5_model = SentenceTransformer(E5_MODEL_NAME, device=DEVICE)

# ------------------------------------------------------------
# Compute lexical + semantic metrics
# ------------------------------------------------------------

summary_rows = []

for exp in experiments:
    label = exp["label"]

    print("\n" + "=" * 90)
    print("Processing:", label)
    print("=" * 90)

    if not exp["found"]:
        summary_rows.append({
            "experiment": label,
            "model_type": exp["model_type"],
            "found": False,
            "num_examples": None,
            "BLEU": None,
            "chrF": None,
            "chrF++": None,
            "E5_large_cosine_mean": None,
            "E5_large_cosine_median": None,
            "E5_large_cosine_std": None,
            "E5_large_cosine_p05": None,
            "E5_large_cosine_p95": None,
            "prediction_file": exp["prediction_file"],
            "metrics_path": exp["metrics_path"],
            "error": exp["error"],
        })
        print("Skipped:", exp["error"])
        continue

    pred_file = Path(exp["prediction_file"])
    df = load_prediction_df(pred_file)

    print("Prediction file:", pred_file)
    print("Rows after deduplication:", len(df))

    # Always compute BLEU / chrF / chrF++ from the CSV.
    # This prevents NaN for baselines and also fixes missing chrF++ in old JSONs.
    mt_scores = compute_mt_metrics_from_df(df)

    print("BLEU:", mt_scores["BLEU"])
    print("chrF:", mt_scores["chrF"])
    print("chrF++:", mt_scores["chrF++"])

    # E5-large semantic similarity
    sims = compute_e5_pairwise_similarity(df, e5_model)
    df["E5_large_cosine"] = sims

    semantic_scores = summarize_similarity(sims)

    semantic_pred_path = SEMANTIC_OUT_DIR / f"semantic_predictions_e5_large_{pred_file.stem}.csv"
    semantic_json_path = SEMANTIC_OUT_DIR / f"semantic_metrics_e5_large_{pred_file.stem}.json"

    df.to_csv(semantic_pred_path, index=False, encoding="utf-8-sig")

    metrics_data = exp.get("metrics_data") or {}

    experiment_name = metrics_data.get(
        "experiment",
        safe_first(df, "experiment_name", label)
    )

    semantic_report = {
        "experiment": experiment_name,
        "display_name": label,
        "model_type": exp["model_type"],

        "checkpoint": metrics_data.get("checkpoint"),
        "best_step": metrics_data.get("best_step"),
        "best_eval_loss": metrics_data.get("best_eval_loss"),

        "semantic_model": E5_MODEL_NAME,
        "semantic_metric": "pairwise cosine similarity between reference_arabic and prediction",
        "num_examples": int(len(df)),
        "unique_source_ids": int(df["source_id"].nunique()),

        **mt_scores,
        **semantic_scores,

        "prediction_file": str(pred_file),
        "semantic_prediction_file": str(semantic_pred_path),
        "source_metrics_path": exp["metrics_path"],

        "per_config": compute_group_metrics(df, "config"),
        "per_dialect": compute_group_metrics(df, "dialect"),
        "per_domain": compute_group_metrics(df, "domain"),
    }

    semantic_json_path.write_text(
        json.dumps(semantic_report, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    summary_rows.append({
        "experiment": label,
        "experiment_name": experiment_name,
        "model_type": exp["model_type"],
        "found": True,
        "num_examples": semantic_report["num_examples"],

        "BLEU": mt_scores["BLEU"],
        "chrF": mt_scores["chrF"],
        "chrF++": mt_scores["chrF++"],

        "E5_large_cosine_mean": semantic_scores["E5_large_cosine_mean"],
        "E5_large_cosine_median": semantic_scores["E5_large_cosine_median"],
        "E5_large_cosine_std": semantic_scores["E5_large_cosine_std"],
        "E5_large_cosine_p05": semantic_scores["E5_large_cosine_p05"],
        "E5_large_cosine_p95": semantic_scores["E5_large_cosine_p95"],

        "best_step": metrics_data.get("best_step"),
        "best_eval_loss": metrics_data.get("best_eval_loss"),
        "checkpoint": metrics_data.get("checkpoint"),

        "prediction_file": str(pred_file),
        "semantic_json": str(semantic_json_path),
        "semantic_predictions": str(semantic_pred_path),
        "error": "",
    })

    print("Mean E5-large cosine:", semantic_scores["E5_large_cosine_mean"])
    print("Saved semantic predictions:", semantic_pred_path)
    print("Saved semantic report:", semantic_json_path)

# ------------------------------------------------------------
# Final comparison table
# ------------------------------------------------------------

semantic_comparison_df = pd.DataFrame(summary_rows)

numeric_cols = [
    "BLEU",
    "chrF",
    "chrF++",
    "E5_large_cosine_mean",
    "E5_large_cosine_median",
    "E5_large_cosine_std",
    "E5_large_cosine_p05",
    "E5_large_cosine_p95",
    "best_eval_loss",
]

for c in numeric_cols:
    if c in semantic_comparison_df.columns:
        semantic_comparison_df[c] = pd.to_numeric(semantic_comparison_df[c], errors="coerce")

semantic_comparison_df = semantic_comparison_df.sort_values(
    by=["found", "E5_large_cosine_mean"],
    ascending=[False, False],
    na_position="last",
).reset_index(drop=True)

comparison_path = SEMANTIC_OUT_DIR / "comparison_semantic_similarity_e5_large_qwen_gemma_baselines.csv"

semantic_comparison_df.to_csv(
    comparison_path,
    index=False,
    encoding="utf-8-sig",
)

print("\n" + "=" * 90)
print("Final Lexical + Semantic Comparison using E5-large")
print("=" * 90)

display_cols = [
    "experiment",
    "model_type",
    "found",
    "num_examples",
    "BLEU",
    "chrF",
    "chrF++",
    "E5_large_cosine_mean",
    "E5_large_cosine_median",
    "E5_large_cosine_std",
    "E5_large_cosine_p05",
    "E5_large_cosine_p95",
    "best_step",
    "best_eval_loss",
    "error",
]

display(semantic_comparison_df[display_cols])

print("\nFiles saved:")
display(
    semantic_comparison_df[
        [
            "experiment",
            "prediction_file",
            "semantic_json",
            "semantic_predictions",
        ]
    ]
)

print("\nSaved final comparison to:")
print(comparison_path)

QWEN_METRICS_PATH: /content/drive/MyDrive/alexandria_qwen35_sft/predictions/full_eval_metrics_qwen35_2b_alexandria_eg_only_context3_all_group_r16_v2_10epochs_best_step600.json
GEMMA_METRICS_PATH: /content/drive/MyDrive/alexandria_qwen35_sft/predictions/full_eval_metrics_gemma4_e2b_it_alexandria_eg_only_context3_fnn_group_r8_prompt_nat_eg_mlp_r8_lr1e5_10epochs_v1_best_step2500.json
BASELINE_PRED_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/baselines/predictions
SEMANTIC_OUT_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/semantic_similarity_e5_large
Using device: cuda
E5 model: intfloat/multilingual-e5-large
Batch size: 8

Experiments to evaluate:
- Qwen3.5-2B LoRA all-r16 best_step600 | found: True | pred: /content/drive/MyDrive/alexandria_qwen35_sft/predictions/full_eval_predictions_qwen35_2b_alexandria_eg_only_context3_all_group_r16_v2_10epochs_best_step600.csv
- Gemma-4-E2B-it FNN-r8 MLP best_step2500 | found: True | pred: /content/drive/MyDrive/alexandria_qwen35_sft/predicti

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

Mean E5-large cosine: 0.9233242767548092
Saved semantic predictions: /content/drive/MyDrive/alexandria_qwen35_sft/semantic_similarity_e5_large/semantic_predictions_e5_large_full_eval_predictions_baseline_no_finetune_qwen35_2b_base_alexandria_eg_only_context3.csv
Saved semantic report: /content/drive/MyDrive/alexandria_qwen35_sft/semantic_similarity_e5_large/semantic_metrics_e5_large_full_eval_predictions_baseline_no_finetune_qwen35_2b_base_alexandria_eg_only_context3.json

Final Lexical + Semantic Comparison using E5-large


,experiment,model_type,found,num_examples,BLEU,chrF,chrF++,E5_large_cosine_mean,E5_large_cosine_median,E5_large_cosine_std,E5_large_cosine_p05,E5_large_cosine_p95,best_step,best_eval_loss,error
0,baseline_no_finetune_gemma4_e2b_it_base_alexan...,baseline_no_finetune,True,1118,14.545134,44.957983,41.905887,0.952464,0.954937,0.023460,0.910387,0.986728,NaN,NaN,
1,Gemma-4-E2B-it FNN-r8 MLP best_step2500,finetuned_lora,True,1118,13.369876,43.155348,40.144052,0.949918,0.951944,0.023773,0.908785,0.984998,2500.0,2.918498,
2,Qwen3.5-2B LoRA all-r16 best_step600,finetuned_lora,True,1118,8.561227,37.912548,34.572649,0.943186,0.944604,0.024783,0.897434,0.980958,600.0,2.015077,
3,baseline_no_finetune_qwen35_2b_base_alexandria...,baseline_no_finetune,True,1118,2.366042,26.516454,23.239397,0.923324,0.927913,0.032006,0.866577,0.968738,NaN,NaN,



Files saved:


,experiment,prediction_file,semantic_json,semantic_predictions
0,baseline_no_finetune_gemma4_e2b_it_base_alexan...,/content/drive/MyDrive/alexandria_qwen35_sft/b...,/content/drive/MyDrive/alexandria_qwen35_sft/s...,/content/drive/MyDrive/alexandria_qwen35_sft/s...
1,Gemma-4-E2B-it FNN-r8 MLP best_step2500,/content/drive/MyDrive/alexandria_qwen35_sft/p...,/content/drive/MyDrive/alexandria_qwen35_sft/s...,/content/drive/MyDrive/alexandria_qwen35_sft/s...
2,Qwen3.5-2B LoRA all-r16 best_step600,/content/drive/MyDrive/alexandria_qwen35_sft/p...,/content/drive/MyDrive/alexandria_qwen35_sft/s...,/content/drive/MyDrive/alexandria_qwen35_sft/s...
3,baseline_no_finetune_qwen35_2b_base_alexandria...,/content/drive/MyDrive/alexandria_qwen35_sft/b...,/content/drive/MyDrive/alexandria_qwen35_sft/s...,/content/drive/MyDrive/alexandria_qwen35_sft/s...



Saved final comparison to:
/content/drive/MyDrive/alexandria_qwen35_sft/semantic_similarity_e5_large/comparison_semantic_similarity_e5_large_qwen_gemma_baselines.csv
